In [16]:
import os
import pandas as pd
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from tensorflow.keras import layers, models


def explore_image_directory(root_dir):
    car_images = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(('.jpg', '.jpeg', '.png')):
                car_images.append(os.path.join(root, file))
    print(f"Найдено {len(car_images)} изображений")
    return car_images[:1000]

def extract_color_from_filename(filename):
    parts = os.path.basename(filename).split('$$')
    if len(parts) >= 4:
        return parts[3]
    return None

def load_and_preprocess_image(image_path):
    try:
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f"Не удалось загрузить изображение: {image_path}")
        img = cv2.resize(img, (100, 100))
        return img
    except Exception as e:
        print(f"Ошибка при обработке изображения {image_path}: {e}")
        return np.zeros((100, 100, 3))

car_images = explore_image_directory('confirmed_fronts')
image_colors = {img_path: extract_color_from_filename(img_path) for img_path in car_images if extract_color_from_filename(img_path)}
df = pd.DataFrame(list(image_colors.items()), columns=['image_path', 'color'])
label_encoder = LabelEncoder()
df['color_encoded'] = label_encoder.fit_transform(df['color'])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_images = np.array([load_and_preprocess_image(img_path) for img_path in train_df['image_path']])
test_images = np.array([load_and_preprocess_image(img_path) for img_path in test_df['image_path']])

model = models.Sequential([
    layers.Input(shape=(100, 100, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_images, train_df['color_encoded'], epochs=100, validation_split=0.2, verbose=0)
predictions = np.argmax(model.predict(test_images), axis=1)
f1 = f1_score(test_df['color_encoded'], predictions, average='macro')
print(f"F1-macro: {f1}")

Найдено 61827 изображений
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
F1-macro: 0.4133584776430809
